[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ELTE-DSED/Intro-Data-Security/blob/main/module_08_defenses/Lab_8a_Differential_Privacy_and_DP_SGD.ipynb)

# **Lab 8a: Differential Privacy and DP-SGD**

**Course:** Introduction to Data Security  
**Module 8:** Defenses  
**Estimated Time:** 90 minutes

---

## Overview

[Differential privacy](https://en.wikipedia.org/wiki/Differential_privacy) (DP) is a framework for measuring the privacy guarantees provided by an algorithm. Through the lens of differential privacy, you can design machine learning algorithms that responsibly train models on private data. 

A model trained with differential privacy should not be affected by any single training example, or small set of training examples, in its data set. This helps mitigate the risk of exposing sensitive training data (such as in Membership Inference or Reconstruction attacks).

### Differentially Private SGD (DP-SGD)

The core approach is to modify the gradients used in Stochastic Gradient Descent (SGD). There are two fundamental modifications made to the vanilla SGD algorithm:

1.  **Sensitivity Bounding (Clipping)**: You must limit how much each individual training point in a minibatch can influence model updates. This is done by *clipping* the L2 norm of each gradient computed on each training point.
2.  **Noise Injection**: Random Gaussian noise is sampled and added to the aggregated clipped gradients. This makes it statistically impossible to determine if a particular data point was included in the training set by observing the model updates.

## Learning Objectives

By the end of this lab, you will understand:

1. **Differential Privacy (DP):** Formal privacy guarantees for ML
2. **DP-SGD:** Gradient clipping + noise injection
3. **Privacy Budget:** How to measure privacy loss
4. **Utility Trade-offs:** Privacy vs model accuracy
5. **Privacy Attacks:** Membership inference risk reduction

## Table of Contents

1. [Baseline Training](#baseline)
2. [Manual DP-SGD Implementation](#dpsgd)
3. [Industry Standard DP-SGD (Opacus)](#opacus)
4. [Sensitivity Analysis: Noise vs Utility](#sensitivity)
5. [Empirical Audit: Membership Inference](#mia)
6. [Exercises](#exercises)

## Setup

In [ ]:
%pip install opacus

In [ ]:
import os
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset
import torchvision.transforms as transforms
from torchvision.datasets import MNIST

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
try:
    plt.style.use('seaborn-v0_8-muted')
except:
    plt.style.use('seaborn-muted')
from sklearn.metrics import roc_auc_score
from dataclasses import dataclass

np.random.seed(42)
torch.manual_seed(42)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
torch.use_deterministic_algorithms(True) 

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

### Load MNIST

In [ ]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

train_dataset = MNIST(root='./data', train=True, download=True, transform=transform)
test_dataset = MNIST(root='./data', train=False, download=True, transform=transform)

train_indices = np.random.choice(len(train_dataset), 1000, replace=False)
test_indices = np.random.choice(len(test_dataset), 500, replace=False)

train_data = Subset(train_dataset, train_indices)
test_data = Subset(test_dataset, test_indices)

train_loader = DataLoader(train_data, batch_size=64, shuffle=True)
test_loader = DataLoader(test_data, batch_size=64, shuffle=False)

print(f"Train: {len(train_data)}, Test: {len(test_data)}")

## Model Definition

In [ ]:
class MLP(nn.Module):
    def __init__(self):
        super(MLP, self).__init__()
        self.flatten = nn.Flatten()
        self.fc1 = nn.Linear(28 * 28, 512)
        self.fc2 = nn.Linear(512, 256)
        self.fc3 = nn.Linear(256, 10)
    
    def forward(self, x):
        x = self.flatten(x)
        x = torch.relu(self.fc1(x))
        x = torch.relu(self.fc2(x))
        x = self.fc3(x)
        return x

In [ ]:
def evaluate(model, loader):
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for data, target in loader:
            data, target = data.to(device), target.to(device)
            output = model(data)
            pred = output.argmax(dim=1)
            correct += (pred == target).sum().item()
            total += target.size(0)
    return 100.0 * correct / total

## Baseline Training (No DP) <a id="baseline"></a>

In [ ]:
def train_baseline(model, loader, epochs=5):
    model.to(device)
    optimizer = optim.SGD(model.parameters(), lr=0.01, momentum=0.9, weight_decay=0)
    criterion = nn.CrossEntropyLoss()
    losses = []
    for epoch in range(epochs):
        epoch_loss = 0
        for data, target in loader:
            data, target = data.to(device), target.to(device)
            optimizer.zero_grad()
            output = model(data)
            loss = criterion(output, target)
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()
        losses.append(epoch_loss / len(loader))
        print(f"Epoch {epoch+1}/{epochs}: Loss={losses[-1]:.4f}")
    return losses

In [ ]:
baseline_model = MLP()
baseline_losses = train_baseline(baseline_model, train_loader, epochs=30)

baseline_train_acc = evaluate(baseline_model, train_loader)
baseline_test_acc = evaluate(baseline_model, test_loader)

print(f"\nBaseline Accuracy:")
print(f"  Train: {baseline_train_acc:.2f}%")
print(f"  Test:  {baseline_test_acc:.2f}%")

## DP-SGD Implementation <a id="dpsgd"></a>

Before we privatize the model, it is crucial to understand the four knobs you will be tuning:

1.  **`max_grad_norm` (Clipping Threshold)**: The maximum L2 norm allowed for any single gradient. It bounds the "sensitivity" of the model to any individual data point.
2.  **`noise_multiplier` ($\sigma$)**: The amount of Gaussian noise added to gradients. More noise = better privacy, but usually lower accuracy.
3.  **`batch_size`**: We must compute gradients for every sample.
4.  **`lr`**: Because DP-SGD gradients are noisy, you often need to use a different learning rate than your baseline. A common strategy is to use a slightly higher learning rate with a scheduler to help the model converge despite the noise.

In [ ]:
@dataclass
class DPConfig:
    max_grad_norm: float = 1.0
    noise_multiplier: float = 1.0  # σ
    epochs: int = 5
    batch_size: int = 64
    lr: float = 0.01

In [ ]:
def dp_sgd_step(model, data, target, config: DPConfig):
    """Single DP-SGD step with per-sample gradient clipping + noise."""
    model.train()
    criterion = nn.CrossEntropyLoss(reduction='none')
    
    data, target = data.to(device), target.to(device)
    batch_size = data.size(0)
    
    # Compute per-sample gradients
    batch_loss = 0.0
    per_sample_grads = []
    for i in range(batch_size):
        model.zero_grad()
        output = model(data[i:i+1])
        loss = criterion(output, target[i:i+1]).mean()
        batch_loss += loss.item()
        loss.backward()
        grads = [p.grad.detach().clone() for p in model.parameters()]
        per_sample_grads.append(grads)
    
    # Clip per-sample gradients
    clipped_grads = []
    for grads in per_sample_grads:
        total_norm = torch.sqrt(sum((g**2).sum() for g in grads))
        clip_factor = min(1.0, config.max_grad_norm / (total_norm + 1e-6))
        clipped = [g * clip_factor for g in grads]
        clipped_grads.append(clipped)
    
    # Aggregate clipped gradients
    agg_grads = []
    for param_i in range(len(clipped_grads[0])):
        stacked = torch.stack([g[param_i] for g in clipped_grads], dim=0)
        agg = stacked.mean(dim=0)
        agg_grads.append(agg)
    
    # Add noise
    noisy_grads = []
    for g in agg_grads:
        noise = torch.randn_like(g) * (config.noise_multiplier * config.max_grad_norm / batch_size)
        noisy_grads.append(g + noise)
    
    # Apply gradients
    for param, grad in zip(model.parameters(), noisy_grads):
        param.grad = grad
    return batch_loss / batch_size

In [ ]:
def train_dp_sgd(model, loader, config: DPConfig):
    optimizer = optim.SGD(model.parameters(), lr=config.lr, momentum=0.9)
    losses = []
    for epoch in range(config.epochs):
        epoch_loss = 0
        for data, target in loader:
            optimizer.zero_grad()
            step_loss = dp_sgd_step(model, data, target, config)
            optimizer.step()
            # Compute loss for logging
            epoch_loss += step_loss
        losses.append(epoch_loss / len(loader))
        print(f"Epoch {epoch+1}/{config.epochs}: Loss={losses[-1]:.4f}")
    return losses

In [ ]:
dp_config = DPConfig(max_grad_norm=1.0, noise_multiplier=0.8, epochs=30, lr=0.02)
dp_model = MLP().to(device)
dp_losses = train_dp_sgd(dp_model, train_loader, dp_config)

dp_train_acc = evaluate(dp_model, train_loader)
dp_test_acc = evaluate(dp_model, test_loader)

print(f"\nDP-SGD Accuracy:")
print(f"  Train: {dp_train_acc:.2f}%")
print(f"  Test:  {dp_test_acc:.2f}%")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(baseline_losses, label='Baseline', color='#e74c3c', alpha=0.5)
axes[0].plot(dp_losses, label='DP-SGD', color='#3498db', linewidth=2)
axes[0].set_title('Training Loss Comparison')
axes[0].set_xlabel('Epoch')
axes[0].legend()
axes[0].grid(alpha=0.3)

axes[1].bar(['Baseline', 'DP-SGD'], [baseline_test_acc, dp_test_acc], color=['#e74c3c', '#3498db'])
axes[1].set_title('Utility Trade-off (Test Accuracy)')
axes[1].set_ylabel('Accuracy (%)')
axes[1].set_ylim(0, 100)
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

## Industry Standard DP-SGD (Opacus) <a id="opacus"></a>

While manual implementation is great for understanding, production environments use libraries like **Opacus** (developed by Meta).

In [ ]:
from opacus import PrivacyEngine
from opacus.utils.batch_memory_manager import BatchMemoryManager

# 1. Initialize model, optimizer, and data loader
opacus_model = MLP().to(device)
opacus_optimizer = optim.SGD(opacus_model.parameters(), lr=0.02, momentum=0.9)
opacus_train_loader = DataLoader(train_data, batch_size=64, shuffle=True)

# 2. Attach Privacy Engine
privacy_engine = PrivacyEngine()
model, optimizer, train_loader = privacy_engine.make_private(
    module=opacus_model,
    optimizer=opacus_optimizer,
    data_loader=opacus_train_loader,
    noise_multiplier=0.8,
    max_grad_norm=1.0,
)

print(f"Privatized model using σ={0.8} and C={1.0}")

### Training with Opacus
Note how the `PrivacyEngine` tracks the privacy budget automatically as we train.

In [ ]:
def train_opacus(model, optimizer, loader, epochs=20):
    model.train()
    criterion = nn.CrossEntropyLoss()
    losses = []
    epsilons = []
    
    # Using BatchMemoryManager is best practice for large batches in DP
    with BatchMemoryManager(
        data_loader=loader, max_physical_batch_size=64, optimizer=optimizer
    ) as memory_safe_loader:
        for epoch in range(epochs):
            epoch_loss = 0
            for data, target in memory_safe_loader:
                data, target = data.to(device), target.to(device)
                optimizer.zero_grad()
                output = model(data)
                loss = criterion(output, target)
                loss.backward()
                optimizer.step()
                epoch_loss += loss.item()
            
            epsilon = privacy_engine.get_epsilon(delta=1e-5)
            losses.append(epoch_loss / len(loader))
            epsilons.append(epsilon)
            if (epoch + 1) % 5 == 0:
                print(f"Epoch {epoch+1}/{epochs}: Loss={losses[-1]:.4f}, ε={epsilon:.2f}")
    return losses, epsilons

In [ ]:
opacus_losses, epsilons = train_opacus(model, optimizer, train_loader, epochs=20)
opacus_test_acc = evaluate(model, test_loader)

plt.figure(figsize=(8, 4))
plt.plot(epsilons, marker='o', color='#2ecc71', linewidth=2)
plt.title('Epsilon (Privacy Budget) Growth Over Epochs')
plt.xlabel('Epoch')
plt.ylabel('ε (at δ=1e-5)')
plt.grid(alpha=0.3)
plt.show()

print(f"\nOpacus DP-SGD Test Accuracy: {opacus_test_acc:.2f}%")

## Sensitivity Analysis: Noise vs Utility <a id="sensitivity"></a>

Differential Privacy is all about the **Trade-off**. If we add more noise ($\sigma$), we get better privacy (lower $\epsilon$), but our accuracy usually drops.

> **Note on Warnings:** At very low noise levels (e.g., $\sigma=0.1$), you may see numerical warnings. This is expected, as the privacy budget becomes mathematically unbounded (infinite) when noise is near-zero.

In [ ]:
import warnings
warnings.filterwarnings("ignore", category=RuntimeWarning)
warnings.filterwarnings("ignore", message=".*Secure RNG turned off.*")
warnings.filterwarnings("ignore", message=".*Using a non-full backward hook.*")

noises = [0.1, 0.5, 1.0, 2.0]
results = []

print("Running Sensitivity Analysis...")
for sigma in noises:
    # Re-init model and engine
    m = MLP().to(device)
    opt = optim.SGD(m.parameters(), lr=0.02, momentum=0.9)
    pe = PrivacyEngine()
    m_priv, opt_priv, dl_priv = pe.make_private(
        module=m, optimizer=opt, data_loader=opacus_train_loader, 
        noise_multiplier=sigma, max_grad_norm=1.0
    )
    
    # Train for fewer epochs for speed
    _ = train_opacus(m_priv, opt_priv, dl_priv, epochs=10)
    acc = evaluate(m_priv, test_loader)
    eps = pe.get_epsilon(delta=1e-5)
    results.append({'noise': sigma, 'accuracy': acc, 'epsilon': eps})
    print(f"σ={sigma}: Acc={acc:.2f}%, ε={eps:.2f}")

res_df = pd.DataFrame(results)

fig, ax1 = plt.subplots(figsize=(8, 4))
ax2 = ax1.twinx()

ax1.plot(res_df['noise'], res_df['accuracy'], marker='o', color='#3498db', label='Accuracy')
ax2.plot(res_df['noise'], res_df['epsilon'], marker='s', color='#e74c3c', label='Epsilon')

ax1.set_xlabel('Noise Multiplier (σ)')
ax1.set_ylabel('Accuracy (%)', color='#3498db')
ax2.set_ylabel('Epsilon (ε)', color='#e74c3c')
plt.title('Sensitivity Analysis: The Privacy-Utility Trade-off')
plt.grid(alpha=0.3)
plt.show()

- As Noise ($\sigma$) increases, Epsilon ($\epsilon$) decreases (better privacy).
- There is usually a point where adding more noise causes the accuracy to drop sharply.
- Professionals look for the 'knee' in this curve, the point where we get the most privacy for the least accuracy loss.

## Empirical Audit: Membership Inference <a id="mia"></a>

Does the mathematical guarantee ($\epsilon$) actually translate to protection against real attacks?

In [ ]:
def get_mia_scores(model, loader):
    model.eval()
    scores = []
    criterion = nn.CrossEntropyLoss(reduction='none')
    with torch.no_grad():
        for data, target in loader:
            data, target = data.to(device), target.to(device)
            output = model(data)
            loss = criterion(output, target)
            # Use negative loss as score (higher = more likely to be in training set)
            scores.extend((-loss).cpu().numpy())
    return np.array(scores)

In [ ]:
# Baseline model MIA
baseline_train_scores = get_mia_scores(baseline_model, train_loader)
baseline_test_scores = get_mia_scores(baseline_model, test_loader)
labels_base = np.concatenate([np.ones(len(baseline_train_scores)), np.zeros(len(baseline_test_scores))])
scores_base = np.concatenate([baseline_train_scores, baseline_test_scores])
auc_base = roc_auc_score(labels_base, scores_base)

# Opacus DP model MIA
opacus_train_scores = get_mia_scores(opacus_model, train_loader)
opacus_test_scores = get_mia_scores(opacus_model, test_loader)
labels_opacus = np.concatenate([np.ones(len(opacus_train_scores)), np.zeros(len(opacus_test_scores))])
scores_opacus = np.concatenate([opacus_train_scores, opacus_test_scores])
auc_opacus = roc_auc_score(labels_opacus, scores_opacus)

# Manual DP model MIA
dp_train_scores = get_mia_scores(dp_model, train_loader)
dp_test_scores = get_mia_scores(dp_model, test_loader)
labels_dp = np.concatenate([np.ones(len(dp_train_scores)), np.zeros(len(dp_test_scores))])
scores_dp = np.concatenate([dp_train_scores, dp_test_scores])
auc_dp = roc_auc_score(labels_dp, scores_dp)

print("\nMembership Inference Audit:")
print(f"  Baseline AUC: {auc_base:.4f}")
print(f"  DP-SGD AUC:   {auc_dp:.4f}")

In this lab, we focused on **DP-SGD**, which is a **Model-level defense**. 

**Summary:** Use DP-SGD when you have sensitive raw data and need the absolute best accuracy for a specific task. Use Synthetic Data when you need to share data with multiple teams for exploratory analysis without risk.

## Exercises <a id="exercises"></a>

### Exercise 1: The Epsilon Budget
Using the Opacus implementation, fix the noise multiplier to $\sigma=1.0$. 
1. Train for 10 epochs and record the final $\epsilon$.
2. Train for 40 epochs and record the final $\epsilon$.

**Question:** Why does privacy leak more as we train for more epochs, even if the noise per step is the same?

### Exercise 2: Replacing BatchNorm
Modify the `MLP` architecture to include `nn.BatchNorm2d` layers. 
1. Try to run the Opacus `make_private` function. What error do you get?
2. Use `opacus.validators.ModuleValidator.fix(model)` to fix the model. What layers did it replace the BatchNorm with?

### Exercise 3: Empirical Audit vs. Theoretical Guarantee
Run the MIA audit on a model trained with $\sigma=0.1$ (weak privacy) and $\sigma=1.5$ (strong privacy).

**Task:** Report the MIA AUC for both. Does the 'empirical' risk (MIA) decrease as the 'theoretical' risk ($\epsilon$) decreases?